# Coupled VarFlExI wake and flow field analysis

This notebook inventories the run and analyzes the selected dynamic `pfield` and companion `staticpfield` particle snapshots. It uses the project postprocessor `fluid/postprocess_particle_fields.jl`, which calls FLOWUnsteady's `computefluiddomain`/GeometricTools routines to evaluate the saved rVPM particle field. It does not edit the original simulation outputs. Derived files are written beneath `analysis/02_flow_field_analysis/`.

Run the notebook cells from top to bottom. Cell 7 writes the cumulative bounds table and JSON; cell 9 runs field reconstruction; the later cells compute diagnostics and plots.

The reconstruction processes 150 selected snapshots: steps 0, 2, ..., 298. Each snapshot uses a cumulative particle bounding box from all dynamic and static particle positions from step 0 through that snapshot, with 32×32×32 cells. The exact 150-snapshot count means step 300 is excluded; including both endpoints would produce 151 snapshots.

**Interpretation limit:** positive Q and irregular time histories are diagnostics, not proof of turbulence or deterministic chaos. This is a finite-domain, kernel-filtered reconstruction from an rVPM simulation, not an independent Eulerian CFD solution.

In [ ]:
from pathlib import Path
import os, re, sys, json, math, subprocess, warnings
import xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt
from scipy.interpolate import RegularGridInterpolator
from IPython.display import display
import yaml

RUN_DIR = Path("/media/dysco/New Volume/Neeraj/VarFlExI_Results/rect_nonlinear")
PARTICLE_DIR = RUN_DIR / "fluid"
ANALYSIS_DIR = RUN_DIR / "analysis" / "02_flow_field_analysis"
DATA_DIR = ANALYSIS_DIR / "data"
FIELD_DIR = ANALYSIS_DIR / "reconstructed_fields" / "steps_0_298_stride_2_32cubed"
SELECTED_DIR = FIELD_DIR / "selected_snapshots"
PLOT_DIR = ANALYSIS_DIR / "plots"
for d in [DATA_DIR, FIELD_DIR, SELECTED_DIR, *[PLOT_DIR / x for x in ["particle_visualization", "vorticity_fields", "velocity_fields", "vortex_structures", "flow_diagnostics", "animations"]]]:
    d.mkdir(parents=True, exist_ok=True)

REPO = Path.cwd()
if not (REPO / "fluid" / "postprocess_particle_fields.jl").is_file(): REPO = REPO.parent
POSTPROCESS = REPO / "fluid" / "postprocess_particle_fields.jl"
JULIA_PROJECT = REPO.parent
FLUID_CONFIG = REPO / "config/fluid_params.yaml"
SOLID_CONFIG = REPO / "config/solid_params.yaml"
COUPLING_CONFIG = REPO / "config/coupling_params.yaml"
CONFIG_ROOT = RUN_DIR / "config" if (RUN_DIR / "config").is_dir() else REPO / "config"
_cfg_fluid = yaml.safe_load((CONFIG_ROOT / "fluid_params.yaml").read_text()) or {}
_cfg_coupling = yaml.safe_load((CONFIG_ROOT / "coupling_params.yaml").read_text()) or {}
RUN_DT = float(_cfg_coupling.get("total_time", _cfg_fluid.get("total_time", 1.0))) / int(_cfg_coupling.get("n_steps", _cfg_fluid.get("nsteps", 1)))

# Cumulative bounds expand with particle-cloud extrema through each selected time.
GRID_CELLS = (32, 32, 32)  # cell counts; each output has 33 nodes per axis
SNAPSHOT_STRIDE = 2
POSTPROCESS_STEPS = list(range(0, 300, SNAPSHOT_STRIDE))  # 0..298: exactly 150 snapshots
KERNEL_TAIL_TOLERANCE = 1.0e-6  # reported for Gaussian-tail truncation context; FLOWUnsteady evaluates its own kernel
PROBE_CHORD_FRACTIONS = (0.25, 0.75)
PROBE_SPAN_FRACTIONS = (0.2, 0.5, 0.8)
print("Run:", RUN_DIR, "exists:", RUN_DIR.is_dir())
print("Original data directory is read-only from this notebook; outputs:", ANALYSIS_DIR)
assert RUN_DIR.is_dir() and PARTICLE_DIR.is_dir() and POSTPROCESS.is_file()

## 1. Snapshot inventory and timestamp checks

Every XMF is parsed for its own `<Time>` value, HDF5 reference, geometry and attribute dataset names/dimensions. Snapshot order is chronological by XMF time, not filename. The supplied XMF files contain integer step indices in `<Time>` (for example, snapshot 669 has time 669), which the project source identifies as a legacy writer behavior. The inventory preserves that raw value and adds a corrected physical time `step*dt` using configuration; it records the source for each timestamp. Since this older run has no saved config snapshot, fallback dt is explicitly unverified. Review it before interpreting seconds.

In [ ]:
def xmf_record(path):
    root = ET.parse(path).getroot()
    time_node = next((e for e in root.iter() if e.tag.endswith("Time")), None)
    raw_time = float(time_node.attrib["Value"]) if time_node is not None and "Value" in time_node.attrib else np.nan
    geom = next((e for e in root.iter() if e.tag.endswith("Geometry")), None)
    geom_item = next((e for e in geom.iter() if e.tag.endswith("DataItem")), None) if geom is not None else None
    ref = (geom_item.text or "").strip() if geom_item is not None else ""
    h5name, geom_ds = ref.split(":", 1) if ":" in ref else ("", "")
    attrs=[]
    for a in (e for e in root.iter() if e.tag.endswith("Attribute")):
        di=next((e for e in a.iter() if e.tag.endswith("DataItem")),None)
        href=(di.text or "").strip() if di is not None else ""
        fn, ds=href.split(":",1) if ":" in href else ("","")
        attrs.append((a.attrib.get("Name",""),a.attrib.get("AttributeType",""),di.attrib.get("Dimensions","") if di is not None else "",fn,ds))
    h5path=(path.parent / h5name).resolve() if h5name else None
    n=int(geom_item.attrib.get("Dimensions","0 3").split()[0]) if geom_item is not None else 0
    step_match=re.search(r"\.(\d+)\.xmf$",path.name)
    step=int(step_match.group(1)) if step_match else -1
    dt=RUN_DT
    # Known legacy XMF writer stored integer steps in Time; project source
    # visualization_metadata.jl documents this. Preserve raw_time and correct it.
    legacy=step>=0 and np.isfinite(raw_time) and dt!=1.0 and np.isclose(raw_time,step,atol=1e-10,rtol=0)
    time_s=step*dt if legacy else raw_time
    return {"path":str(path),"population":"staticpfield" if "staticpfield" in path.name.lower() else "pfield",
            "step":step,"time_s_xmf":raw_time,"time_s":time_s,"time_source":"legacy_step_times_dt" if legacy else "xmf" if np.isfinite(raw_time) else "step_times_dt",
            "h5_path":str(h5path) if h5path else "","h5_exists":bool(h5path and h5path.is_file()),
            "geometry_dataset":geom_ds,"particle_count_xmf":n,"geometry_dimensions":geom_item.attrib.get("Dimensions","") if geom_item is not None else "","file_size_xmf_bytes":path.stat().st_size,
            "attributes":attrs}

xmf_files=sorted(PARTICLE_DIR.glob("*.xmf"))
records=[xmf_record(p) for p in xmf_files if re.search(r"(?:_staticpfield|_pfield)\.\d+\.xmf$",p.name)]
records.sort(key=lambda r:(np.inf if not np.isfinite(r["time_s"]) else r["time_s"],r["population"],r["step"]))
inv=[]
for r in records:
    if r["h5_exists"]:
        with h5py.File(r["h5_path"],"r") as h:
            r["file_size_h5_bytes"]=Path(r["h5_path"]).stat().st_size
            r["h5_dataset_keys"]=list(h.keys())
            r["missing_attribute_datasets"]=[a[4] for a in r["attributes"] if a[4] not in h]
            r["particle_count_h5"]=int(h[r["geometry_dataset"]].shape[0]) if r["geometry_dataset"] in h else -1
    else:
        r["file_size_h5_bytes"]=0; r["h5_dataset_keys"]=[]; r["missing_attribute_datasets"]=[a[4] for a in r["attributes"]]; r["particle_count_h5"]=-1
    inv.append({**{k:v for k,v in r.items() if k!="attributes"},"attributes_json":json.dumps([{"name":a[0],"type":a[1],"dimensions":a[2],"h5_file":a[3],"dataset":a[4]} for a in r["attributes"]])})
inventory=pd.DataFrame(inv)
subset_records=[r for r in records if int(r["step"]) in POSTPROCESS_STEPS and r["h5_exists"]]
source_records=[r for r in records if 0 <= int(r["step"]) <= POSTPROCESS_STEPS[-1] and r["h5_exists"]]
inventory.to_csv(DATA_DIR/"particle_snapshot_inventory.csv",index=False)
display(inventory.groupby("population").agg(snapshots=("step","count"),first_time_s=("time_s","min"),last_time_s=("time_s","max"),missing_h5=("h5_exists",lambda x:int((~x).sum()))))
print("timestamp duplicates:",inventory.duplicated(["population","time_s"],keep=False).sum(),"invalid H5 references:",int((~inventory.h5_exists).sum()))
display(inventory.head(3)); display(inventory.tail(3))

## 2. Attribute meanings and solver provenance

The XMF labels distinguish a vector `Gamma` (three components) from scalar `circulation`; do not conflate or scalar-sum them. FLOWVPM's particle state stores position `X`, vector vortex strength `Gamma`, core width `sigma`, and volume-like scalar `vol`; the project XMF writer exposes these with `velocity`, `vorticity`, and velocity-gradient vectors. Units below follow the FLOWVPM definitions and project nondimensionalization; inspect the configuration's length/time scales before attaching units to every exported quantity.

The project postprocessor is the authoritative field evaluator for this run: `fluid/postprocess_particle_fields.jl` loads dynamic and matching static snapshots and calls `uns.computefluiddomain`. FLOWUnsteady documents `U` as particle-induced velocity when `add_Uinf=false`; `W` stores particle vorticity at the evaluation nodes, computed from particle velocity gradients. The utility saves `Uinduced` before adding the configured ambient/gust, saves that ambient as `Uinf`, and stores their sum as total `U`. We retain these distinctions in the output fields. Static particles are included when their matching files exist; the code calls `vpm.read!` on dynamic with overwrite and appends static, so they are not double-counted as the same population.

The active FLOWVPM default particle kernel is Gaussian error-function (`zeta_gauserf`, `g_gauserf`) with normalized radius `r=|x-Xp|/sigma_p`: `zeta(r)=(2*pi)^(-3/2)*exp(-r^2/2)` and `g(r)=erf(r/sqrt(2))-sqrt(2/pi)*r*exp(-r^2/2)`. The `zeta` basis is normalized to unit integral and `g` is the corresponding regularized Biot-Savart factor. Each particle uses its own `sigma`; the kernel is smooth and has nonzero tails, so it has no compact support. The Julia library postprocessor evaluates that kernel directly with its supported FMM/GeometricTools route. We do not substitute a Gaussian approximation or use nearest-particle interpolation. Source references to inspect: `FLOWVPM.jl-master/src/FLOWVPM_kernel.jl`, `FLOWVPM_particlefield.jl`, and `/home/dysco/FLOWUnsteady/leapfrogging/utilities_fluiddomain.jl` (`computefluiddomain`).

Known dimensional interpretation from the FLOWVPM state definitions: `X` and `sigma` are lengths (m), `Gamma` is a vector particle strength (m²/s in the project's SI configuration), `vol` is particle volume (m³), `velocity` is m/s, particle `vorticity` and velocity gradients are s⁻¹. The separate scalar `circulation` is a distinct per-particle bookkeeping value documented by FLOWVPM and is not summed as the vector strength.

FLOWVPM serializes `velocity_gradient_x/y/z` from `J_INDEX[1:3]`, `[4:6]`, `[7:9]`. `get_W` uses these components in the curl identities; the three saved vectors are the derivative-coordinate columns of `A[i,j]=∂u_i/∂x_j`, so reconstruct `A` by stacking the x, y, z datasets on the last axis. The particle snapshot summary cell prints these values and shapes from every available snapshot.

The notebook reports Γ-vs-ωV diagnostics but does not assume equality: `Gamma` is stored as the three-component vector vortex strength, while `circulation` is a separate scalar field. `sigma` is the core width and `vol` the particle volume measure in saved state. We check the actual numerical relationship before interpreting it.

In [ ]:
# Inspect representative source snapshots at the start, middle and end of the selected interval.
sample_steps=[POSTPROCESS_STEPS[0],POSTPROCESS_STEPS[len(POSTPROCESS_STEPS)//2],POSTPROCESS_STEPS[-1]]
sample_records=[r for r in source_records if int(r["step"]) in sample_steps]
all_names=sorted({a[0] for r in sample_records for a in r["attributes"]})
field_rows=[]
for name in all_names:
    shapes=[]; mins=[]; maxs=[]; invalid=[]; counts=[]
    for r in sample_records:
        attr=next((a for a in r["attributes"] if a[0]==name),None)
        if not attr or not r["h5_exists"]: continue
        with h5py.File(r["h5_path"],"r") as h:
            ds=attr[4]
            if ds not in h: continue
            a=np.asarray(h[ds]); finite=np.isfinite(a)
            shapes.append(str(a.shape)); counts.append(a.size); invalid.append(100*(1-finite.mean()))
            if finite.any(): mins.append(np.nanmin(a)); maxs.append(np.nanmax(a))
    field_rows.append({"attribute":name,"sample_shape":shapes[0] if shapes else "","shape_consistent":len(set(shapes))<=1,
                       "sample_min":min(mins) if mins else np.nan,"sample_max":max(maxs) if maxs else np.nan,
                       "invalid_percent_min":min(invalid) if invalid else np.nan,"invalid_percent_max":max(invalid) if invalid else np.nan,
                       "sampled_datasets":len(shapes)})
field_summary=pd.DataFrame(field_rows)
display(field_summary)
print("Numerical attribute summary sampled at steps:",sample_steps)

# Full-array sample per population at the first selected step: shapes, ranges,
# invalid fraction, and Gamma~omega*vol check.
for pop in ["pfield","staticpfield"]:
    r=next((x for x in sample_records if x["population"]==pop and int(x["step"])==POSTPROCESS_STEPS[0] and x["h5_exists"]),None)
    if r is None: print(pop,"no step-0 snapshot"); continue
    print("\n",pop,r["path"])
    with h5py.File(r["h5_path"],"r") as h:
        for key in h.keys():
            a=np.asarray(h[key]); finite=np.isfinite(a)
            print(key,a.shape,"min/max",(float(np.nanmin(a)),float(np.nanmax(a))) if finite.any() else (np.nan,np.nan),"invalid_%",100*(1-finite.mean()))
        if all(k in h for k in ["Gamma","vorticity","vol"]):
            G=np.asarray(h["Gamma"]); w=np.asarray(h["vorticity"]); vol=np.asarray(h["vol"]).reshape(-1)
            candidate=w*vol[:,None] if w.ndim==2 and w.shape[-1]==3 else None
            if candidate is not None:
                rel=np.linalg.norm(G-candidate)/(np.linalg.norm(G)+1e-30)
                print("relative ||Gamma - vorticity*vol||:",rel,"(diagnostic only; verify convention/source before interpreting)")


## 3. Particle diagnostics and cumulative grid bounds

The grid box at each selected time is computed from the minimum and maximum x/y/z particle positions in both dynamic and static populations from step 0 through that output step. A small padding is added, and the undeformed body envelope is included. The grid has 32×32×32 cells at every output time; physical spacing can vary as the cumulative box expands. Bounds are saved for reproducibility.


In [ ]:
# Particle diagnostics and cumulative bounds for the requested 0..298 subset.
# Each step's extrema combine dynamic and static particles. Cumulative extrema
# include every source snapshot from step 0 up through the output step.
particle_rows=[]
step_extrema={}
for r in source_records:
    with h5py.File(r["h5_path"],"r") as h:
        X=np.asarray(h[r["geometry_dataset"]],float)
        row={"population":r["population"],"step":r["step"],"time_s":r["time_s"],"n_particles":len(X)}
        lo=np.nanmin(X,axis=0); hi=np.nanmax(X,axis=0)
        row.update({f"{axis}_{stat}_m":float(val) for axis,idx in zip("xyz",range(3)) for stat,val in [("min",lo[idx]),("max",hi[idx])]})
        if "sigma" in h:
            sig=np.asarray(h["sigma"],float).reshape(-1)
            for stat,val in [("min",np.nanmin(sig)),("median",np.nanmedian(sig)),("max",np.nanmax(sig))]: row[f"sigma_{stat}_m"]=float(val)
        if "Gamma" in h:
            G=np.asarray(h["Gamma"],float); gm=np.linalg.norm(G,axis=1)
            for j,c in enumerate("xyz"): row[f"Gamma_sum_{c}"]=float(np.nansum(G[:,j]))
            row["Gamma_mag_mean"]=float(np.nanmean(gm)); row["Gamma_mag_max"]=float(np.nanmax(gm))
        particle_rows.append(row)
        if r["step"] not in step_extrema:
            step_extrema[r["step"]]=[lo.copy(),hi.copy()]
        else:
            step_extrema[r["step"]][0]=np.minimum(step_extrema[r["step"]][0],lo)
            step_extrema[r["step"]][1]=np.maximum(step_extrema[r["step"]][1],hi)
particle_diag=pd.DataFrame(particle_rows).sort_values(["step","population"])
particle_diag.to_csv(DATA_DIR/"particle_diagnostics_steps_0_298.csv",index=False)
display(particle_diag.groupby("population").agg({"n_particles":["min","median","max"],"sigma_median_m":["min","median","max"]}))

solid_cfg_path=RUN_DIR/"config"/"solid_params.yaml"
fluid_cfg_path=RUN_DIR/"config"/"fluid_params.yaml"
if not solid_cfg_path.exists(): solid_cfg_path=SOLID_CONFIG
if not fluid_cfg_path.exists(): fluid_cfg_path=FLUID_CONFIG
solid_cfg=yaml.safe_load(solid_cfg_path.read_text()) or {}
fluid_cfg=yaml.safe_load(fluid_cfg_path.read_text()) or {}
span=float(solid_cfg.get("span",0.96)); chord=max(float(solid_cfg.get("root_chord",0.24)),float(solid_cfg.get("tip_chord",0.24)))
median_sigma=float(particle_diag.sigma_median_m.median())
# Ensure every integer step needed for cumulative bounds is present in both populations where expected.
missing_source_steps=sorted(set(range(POSTPROCESS_STEPS[-1]+1))-set(step_extrema))
if missing_source_steps: raise ValueError(f"Missing particle geometry snapshots needed for cumulative bounds: {missing_source_steps[:20]}")
missing_populations={step:sorted({"pfield","staticpfield"}-{r["population"] for r in source_records if int(r["step"])==step})
                     for step in range(POSTPROCESS_STEPS[-1]+1)}
missing_populations={step:pop for step,pop in missing_populations.items() if pop}
if missing_populations: raise ValueError(f"Missing particle populations needed for bounds: {list(missing_populations.items())[:10]}")

bounds_rows=[]; bounds_map={}
cum_lo=np.array([np.inf]*3); cum_hi=np.array([-np.inf]*3)
for source_step in range(POSTPROCESS_STEPS[-1]+1):
    lo,hi=step_extrema[source_step]
    cum_lo=np.minimum(cum_lo,lo); cum_hi=np.maximum(cum_hi,hi)
    if source_step not in POSTPROCESS_STEPS: continue
    extent=np.maximum(cum_hi-cum_lo, 1e-9)
    padding=np.maximum(0.02*extent, np.array([0.02*chord,0.01*span,0.01*span]))
    box_lo=cum_lo-padding; box_hi=cum_hi+padding
    # Include the undeformed wing footprint while retaining cumulative particle extrema.
    box_lo[0]=min(box_lo[0],-0.1*chord); box_hi[0]=max(box_hi[0],chord+0.02*chord)
    box_lo[1]=min(box_lo[1],-0.01*span); box_hi[1]=max(box_hi[1],span+0.01*span)
    vals=[float(box_lo[0]),float(box_hi[0]),float(box_lo[1]),float(box_hi[1]),float(box_lo[2]),float(box_hi[2])]
    bounds_map[str(source_step)]=vals
    bounds_rows.append({"step":source_step,"time_s":source_step*RUN_DT,
        "particle_x_min_m":float(cum_lo[0]),"particle_x_max_m":float(cum_hi[0]),
        "particle_y_min_m":float(cum_lo[1]),"particle_y_max_m":float(cum_hi[1]),
        "particle_z_min_m":float(cum_lo[2]),"particle_z_max_m":float(cum_hi[2]),
        "x0_m":vals[0],"x1_m":vals[1],"y0_m":vals[2],"y1_m":vals[3],"z0_m":vals[4],"z1_m":vals[5]})

bounds_json=DATA_DIR/"cumulative_bounds_by_step.json"
bounds_json.write_text(json.dumps(bounds_map,indent=2)+"\n")
bounds_table=pd.DataFrame(bounds_rows)
bounds_table.to_csv(DATA_DIR/"cumulative_bounds_by_step.csv",index=False)
BOUNDS=tuple((bounds_table.iloc[-1][f"{a}0_m"],bounds_table.iloc[-1][f"{a}1_m"]) for a in "xyz")
spacing=np.array([(b-a)/n for (a,b),n in zip(BOUNDS,GRID_CELLS)])
print(f"Selected outputs: {len(POSTPROCESS_STEPS)} at steps {POSTPROCESS_STEPS[0]}..{POSTPROCESS_STEPS[-1]} (stride {SNAPSHOT_STRIDE})")
print("Latest cumulative bounds x/y/z:",BOUNDS,"m")
print("Latest-grid cell spacing:",spacing,"m; dx/median sigma:",spacing/median_sigma)
print("Per-step bounds map:",bounds_json)
display(bounds_table.iloc[[0,1,2,-3,-2,-1]])


## 4. Reconstruct the 150 selected snapshots with FLOWUnsteady

The grid uses 32×32×32 cells. Its bounds at each output time are the cumulative particle extrema through that time, so the box can expand as the wake grows. `computefluiddomain` evaluates particle-induced velocity and velocity-gradient-derived vorticity on that grid; the existing project driver includes matching static particles and adds the configured ambient/gust once, making its saved `U` total velocity and its `W` the particle-induced vorticity. Output goes only into this analysis directory. Grid spacing relative to median sigma is printed above; if it is much larger than 1, these coarse fields cannot resolve individual cores.

Configuration caution: this historical run has no copied `config/` directory in the supplied results. The notebook uses the current repository YAML as an explicit fallback and labels it unverified for historical settings. The XMF timestamps are used by the postprocessor. Do not treat fallback freestream values as run provenance.

In [ ]:
steps=POSTPROCESS_STEPS
assert len(steps)==150 and steps==list(range(0,300,2))
nx,ny,nz=GRID_CELLS
cmd=["julia",f"--project={JULIA_PROJECT}",str(POSTPROCESS),"--input",str(PARTICLE_DIR),"--output",str(FIELD_DIR),
     "--steps",",".join(map(str,steps)),"--nx",str(nx),"--ny",str(ny),"--nz",str(nz),
     "--bounds-file",str(DATA_DIR/"cumulative_bounds_by_step.json"),"--include-static","true","--add-j","false","--add-wapprox","false",
     "--fluid-config",str(fluid_cfg_path),"--solid-config",str(solid_cfg_path),"--coupling-config",str(COUPLING_CONFIG)]
print("Snapshots to reconstruct:",len(steps),"; first/last step:",steps[0],steps[-1],"; stride:",SNAPSHOT_STRIDE,"; cells:",GRID_CELLS,"; output:",FIELD_DIR)
print("Command:"," ".join(cmd))
subprocess.run(cmd,check=True,cwd=REPO)


In [ ]:
# The six visualization times are selected from the already reconstructed subset.
selected_steps=[steps[k] for k in np.unique(np.rint(np.linspace(0,len(steps)-1,6)).astype(int))]
print("Visualization steps (already reconstructed):",selected_steps)


## 5. Reconstructed-field diagnostics, fixed probes, and validation

Run this after the full reconstruction cell. The fields come from FLOWUnsteady/GeometricTools rather than a Python nearest-particle approximation. `Uinduced` is particle induced, `Uinf` is configured ambient/gust, and `U` is their total. The vorticity field used below is the standard FLOWUnsteady `W` field, computed from the particle-induced velocity gradients. Diagnostics use the physical uniform grid and retain the first saved transient. Curl/divergence/Q use interior finite differences; finite-domain truncation, grid spacing, and FMM/postprocessing tolerance can all contribute to residuals. The circulation integral is compared componentwise to particle `Gamma` and may differ when kernel support reaches outside the grid.

The Eulerian output is based on the combined dynamic plus static field because that is how the production postprocessor assembles the full physical field. The source snapshots are still inventoried and summarized separately.

In [ ]:
def read_grid_frame(xmf_path):
    rec=xmf_record(xmf_path); hpath=Path(rec["h5_path"])
    root=ET.parse(xmf_path).getroot(); attrs={}
    for a in (e for e in root.iter() if e.tag.endswith("Attribute")):
        di=next((e for e in a.iter() if e.tag.endswith("DataItem")),None)
        if di is None or not di.text or ":" not in di.text: continue
        _,ds=di.text.strip().split(":",1); attrs[a.attrib.get("Name","")]=ds
    with h5py.File(hpath,"r") as h:
        xyz=np.asarray(h[rec["geometry_dataset"]],float)
        dims=[len(np.unique(np.round(xyz[:,i],12))) for i in range(3)]
        nxn,nyn,nzn=dims
        if nxn*nyn*nzn!=len(xyz): raise ValueError(f"Nonuniform/unstructured output grid: {xmf_path}, dimensions={dims}")
        order=np.lexsort((xyz[:,0],xyz[:,1],xyz[:,2]))
        axes=[np.unique(np.round(xyz[:,i],12)) for i in range(3)]
        fields={}
        for name,ds in attrs.items():
            if ds in h:
                a=np.asarray(h[ds])[order]
                fields[name]=a.reshape((nzn,nyn,nxn)+a.shape[1:])
    x,y,z=axes
    return float(rec["time_s"]),x,y,z,fields

field_files=sorted(FIELD_DIR.glob("*_fluiddomain.*.xmf"),key=lambda p:int(re.search(r"\.(\d+)\.xmf$",p.name).group(1)))
print("Reconstructed field snapshots:",len(field_files))
assert field_files, "Run reconstruction cell first"

# Nominal fixed probes relative to the undeformed body; interpolate total U at fixed positions.
probe_points=[]
for eta in PROBE_SPAN_FRACTIONS:
    for xi in PROBE_CHORD_FRACTIONS:
        probe_points.append((xi*chord,eta*span,0.0))
probe_rows=[]; euler_rows=[]; validation_rows=[]; global_omega_signed_p998=[]; global_omega_mag_p998=[]; global_q_p995=[]
for i,path in enumerate(field_files):
    time,x,y,z,F=read_grid_frame(path)
    U=np.asarray(F.get("U")); Uind=np.asarray(F.get("Uinduced",U)); Uinf=np.asarray(F.get("Uinf",np.zeros_like(U))); W=np.asarray(F["W"])
    if U.ndim!=4 or U.shape[-1]<3 or W.ndim!=4 or W.shape[-1]<3:
        raise ValueError(f"Expected U and reconstructed vorticity fields in {path}; found {F.keys()}")
    dx,dy,dz=float(np.mean(np.diff(x))),float(np.mean(np.diff(y))),float(np.mean(np.diff(z)))
    # Interior derivatives. Array layout is z,y,x,component.
    grad=np.empty(U.shape[:3]+(3,3),float)
    for comp in range(3):
        gz,gy,gx=np.gradient(Uind[...,comp],dz,dy,dx,edge_order=2)
        grad[...,comp,0]=gx; grad[...,comp,1]=gy; grad[...,comp,2]=gz
    curl=np.stack((grad[...,2,1]-grad[...,1,2],grad[...,0,2]-grad[...,2,0],grad[...,1,0]-grad[...,0,1]),axis=-1)
    div=grad[...,0,0]+grad[...,1,1]+grad[...,2,2]
    S=0.5*(grad+np.swapaxes(grad,-1,-2)); Om=0.5*(grad-np.swapaxes(grad,-1,-2))
    qcrit=0.5*(np.sum(Om*Om,axis=(-1,-2))-np.sum(S*S,axis=(-1,-2)))
    interior=(slice(1,-1),slice(1,-1),slice(1,-1))
    wi=W[interior]; ci=curl[interior]; gi=grad[interior]; qi=qcrit[interior]; ui=U[interior]
    vol=dx*dy*dz; wmag=np.linalg.norm(wi,axis=-1); qpos=qi[qi>0]
    global_omega_signed_p998.append(float(np.nanpercentile(np.abs(W[...,1]),99.8)))
    global_omega_mag_p998.append(float(np.nanpercentile(np.linalg.norm(W,axis=-1),99.8)))
    global_q_p995.append(float(np.nanpercentile(qpos,99.5)) if len(qpos) else 0.0)
    row={"time_s":time,"step":int(re.search(r"\.(\d+)\.xmf$",path.name).group(1)),"grid_dx_m":dx,"grid_dy_m":dy,"grid_dz_m":dz,
         "vorticity_rms_s_inv":float(np.sqrt(np.mean(wmag*wmag))),"vorticity_max_s_inv":float(np.max(wmag)),
         "vorticity_p99_s_inv":float(np.percentile(wmag,99)),"resolved_enstrophy":float(0.5*np.sum(wmag*wmag)*vol),
         "Q_positive_volume_fraction":float(np.mean(qi>0)),"Q_positive_max_s2":float(np.max(qpos)) if len(qpos) else 0.0,
         "strain_frobenius_rms_s_inv":float(np.sqrt(np.mean(np.sum(S[interior]**2,axis=(-1,-2))))),
         "rotation_frobenius_rms_s_inv":float(np.sqrt(np.mean(np.sum(Om[interior]**2,axis=(-1,-2))))),
         "velocity_total_rms_m_s":float(np.sqrt(np.mean(np.sum(ui*ui,axis=-1)))),
         "velocity_disturbance_energy_J_per_density":float(0.5*np.sum(np.sum((U-Uinf)[interior]**2,axis=-1))*vol),
         "velocity_induced_rms_m_s":float(np.sqrt(np.mean(np.sum(Uind[interior]**2,axis=-1))))}
    euler_rows.append(row)
    # Vector integral and pointwise curl-vorticity consistency, component by component.
    step_i=int(re.search(r"\.(\d+)\.xmf$",path.name).group(1))
    particle_gamma=np.zeros(3)
    for pop in ("pfield","staticpfield"):
        src=next((r for r in records if r["population"]==pop and r["step"]==step_i and r["h5_exists"]),None)
        if src is not None:
            with h5py.File(src["h5_path"],"r") as h:
                if "Gamma" in h: particle_gamma += np.nansum(np.asarray(h["Gamma"],float),axis=0)
    for j,c in enumerate("xyz"):
        grid_integral=float(np.sum(W[...,j])*vol)
        row[f"vorticity_integral_{c}"]=grid_integral
        row[f"particle_Gamma_sum_{c}"]=float(particle_gamma[j])
        row[f"vorticity_integral_abs_error_{c}"]=abs(grid_integral-particle_gamma[j])
        row[f"vorticity_integral_relative_error_{c}"]=abs(grid_integral-particle_gamma[j])/(abs(particle_gamma[j])+1e-30)
        row[f"curl_integral_{c}"]=float(np.sum(curl[...,j])*vol)
    row["curl_vorticity_relative_l2"]=float(np.linalg.norm(ci-wi)/(np.linalg.norm(wi)+1e-30))
    row["velocity_divergence_relative_l2"]=float(np.linalg.norm(div[interior])/(np.linalg.norm(gi)+1e-30))
    validation_rows.append(row.copy())
    # Fixed Eulerian probes on the total velocity field, interpolation in z,y,x order.
    interp=RegularGridInterpolator((z,y,x),U,bounds_error=False,fill_value=np.nan)
    for pid,point in enumerate(probe_points):
        val=interp([[point[2],point[1],point[0]]])[0]
        probe_rows.append({"time_s":time,"probe":f"xi{point[0]/chord:.2f}_eta{point[1]/span:.2f}","x_m":point[0],"y_m":point[1],"z_m":point[2],"u_total":val[0],"v_total":val[1],"w_total":val[2],"speed_total":np.linalg.norm(val)})
    if i%25==0: print(f"processed diagnostic {i+1}/{len(field_files)} at t={time:g}s")

euler=pd.DataFrame(euler_rows).sort_values("time_s"); euler.to_csv(DATA_DIR/"flow_field_diagnostics.csv",index=False)
probes=pd.DataFrame(probe_rows).sort_values("time_s"); probes.to_csv(DATA_DIR/"eulerian_probe_history.csv",index=False)
pd.DataFrame(validation_rows).to_csv(DATA_DIR/"reconstruction_validation.csv",index=False)
GLOBAL_OMEGA_Y_LIMIT=max(global_omega_signed_p998); GLOBAL_OMEGA_MAG_LIMIT=max(global_omega_mag_p998); GLOBAL_Q_THRESHOLD=max(global_q_p995)
print("Full-record fixed color limits: |omega_y| <=",GLOBAL_OMEGA_Y_LIMIT,"|omega| <=",GLOBAL_OMEGA_MAG_LIMIT,"global Q threshold:",GLOBAL_Q_THRESHOLD)
display(euler.head()); display(euler[["curl_vorticity_relative_l2","velocity_divergence_relative_l2"]].describe())

## 6. Time histories and common-scale midspan wake views

The primary x-z slice is at midspan. Signed `W_y` is shown with a symmetric scale, and vorticity magnitude uses one global nonnegative scale. A shared fixed grid and color range prevent per-frame rescaling from masking changes in amplitude.

In [ ]:
fig,axes=plt.subplots(3,1,figsize=(11,12),sharex=True)
for col,unit in [("resolved_enstrophy","m^3/s^2"),("vorticity_rms_s_inv","1/s"),("Q_positive_volume_fraction","fraction")]:
    axes[0].plot(euler.time_s,euler.resolved_enstrophy); axes[1].plot(euler.time_s,euler.vorticity_rms_s_inv); axes[2].plot(euler.time_s,euler.Q_positive_volume_fraction)
    break
axes[0].set_ylabel("Resolved enstrophy"); axes[1].set_ylabel("Vorticity RMS (1/s)"); axes[2].set_ylabel("Positive-Q volume fraction"); axes[2].set_xlabel("Time (s)")
for ax in axes: ax.grid(alpha=.25)
fig.suptitle("Eulerian flow diagnostics, steps 0–298"); fig.tight_layout(); fig.savefig(PLOT_DIR/"flow_diagnostics"/"eulerian_diagnostics_vs_time.png",dpi=160); plt.show()

fig,ax=plt.subplots(figsize=(10,4))
for probe,part in probes.groupby("probe"): ax.plot(part.time_s,part.speed_total,label=probe)
ax.set(xlabel="Time (s)",ylabel="Total velocity magnitude (m/s)",title="Fixed Eulerian probe histories"); ax.grid(alpha=.25); ax.legend(ncol=3)
fig.tight_layout(); fig.savefig(PLOT_DIR/"flow_diagnostics"/"eulerian_probe_history.png",dpi=160); plt.show()

# Select six actual timestamps spanning the selected interval, no transient removal.
selected_idx=np.unique(np.rint(np.linspace(0,len(field_files)-1,6)).astype(int))
selected_paths=[field_files[k] for k in selected_idx]
# Global scales from the selected/full record's sampled diagnostics; use all-frame maxima via fields if already processed.
frames=[]
for path in selected_paths:
    time,x,y,z,F=read_grid_frame(path); W=F["W"]; U=F["U"]
    iy=int(np.argmin(np.abs(y-span/2)))
    frames.append((time,x,z,W[:,iy,:,:],U[:,iy,:,:]))
max_abs=GLOBAL_OMEGA_Y_LIMIT
max_mag=GLOBAL_OMEGA_MAG_LIMIT
for k,(time,x,z,Wmid,Umid) in enumerate(frames):
    mag=np.linalg.norm(Wmid,axis=-1)
    fig,ax=plt.subplots(figsize=(10,4.5)); im=ax.pcolormesh(x,z,Wmid[...,1],shading="auto",cmap="RdBu_r",vmin=-max_abs,vmax=max_abs)
    ax.set(xlabel="x chord/wake (m)",ylabel="z (m)",title=f"Midspan signed vorticity $\omega_y$, t={time:.5g} s"); fig.colorbar(im,ax=ax,label="1/s")
    fig.tight_layout(); fig.savefig(PLOT_DIR/"vorticity_fields"/f"midspan_omega_y_{k:02d}.png",dpi=160); plt.show()
    fig,ax=plt.subplots(figsize=(10,4.5)); im=ax.pcolormesh(x,z,mag,shading="auto",cmap="magma",vmin=0,vmax=max_mag)
    ax.set(xlabel="x chord/wake (m)",ylabel="z (m)",title=f"Midspan vorticity magnitude, t={time:.5g} s"); fig.colorbar(im,ax=ax,label="1/s")
    fig.tight_layout(); fig.savefig(PLOT_DIR/"vorticity_fields"/f"midspan_omega_mag_{k:02d}.png",dpi=160); plt.show()
    speed=np.linalg.norm(Umid,axis=-1)
    fig,ax=plt.subplots(figsize=(10,4.5)); im=ax.pcolormesh(x,z,speed,shading="auto",cmap="viridis")
    stride=max(1,len(x)//24); zz,xx=np.meshgrid(z,x,indexing="ij")
    ax.quiver(xx[::stride,::stride],zz[::stride,::stride],Umid[::stride,::stride,0],Umid[::stride,::stride,2],color="white",scale=None)
    ax.set(xlabel="x chord/wake (m)",ylabel="z (m)",title=f"Midspan total velocity and (u,w), t={time:.5g} s"); fig.colorbar(im,ax=ax,label="speed (m/s)")
    fig.tight_layout(); fig.savefig(PLOT_DIR/"velocity_fields"/f"midspan_velocity_{k:02d}.png",dpi=160); plt.show()

# Optional full 3D Q isosurfaces, with one global threshold derived from all snapshots.
try:
    import pyvista as pv
    for k,path in enumerate(selected_paths):
        time,x,y,z,F=read_grid_frame(path); Ui=np.asarray(F.get("Uinduced",F["U"]))
        grad=np.empty(Ui.shape[:3]+(3,3),float)
        for comp in range(3):
            gz,gy,gx=np.gradient(Ui[...,comp],np.mean(np.diff(z)),np.mean(np.diff(y)),np.mean(np.diff(x)),edge_order=2)
            grad[...,comp,0]=gx; grad[...,comp,1]=gy; grad[...,comp,2]=gz
        Ss=0.5*(grad+np.swapaxes(grad,-1,-2)); Os=0.5*(grad-np.swapaxes(grad,-1,-2))
        qfield=0.5*(np.sum(Os*Os,axis=(-1,-2))-np.sum(Ss*Ss,axis=(-1,-2)))
        grid=pv.ImageData(dimensions=(len(x),len(y),len(z)),spacing=(float(np.mean(np.diff(x))),float(np.mean(np.diff(y))),float(np.mean(np.diff(z)))),origin=(float(x[0]),float(y[0]),float(z[0])))
        grid.point_data["Q_s-2"]=qfield.ravel(order="C")
        grid.point_data["vorticity_magnitude_s-1"]=np.linalg.norm(F["W"],axis=-1).ravel(order="C")
        surf=grid.contour([GLOBAL_Q_THRESHOLD],scalars="Q_s-2")
        if surf.n_points:
            pl=pv.Plotter(off_screen=True,window_size=(1100,800)); pl.add_mesh(surf,scalars="vorticity_magnitude_s-1",cmap="magma",show_scalar_bar=True)
            pl.add_axes(); pl.add_text(f"positive-Q vortex structures, t={time:.5g}s, Q>{GLOBAL_Q_THRESHOLD:.3g}")
            pl.camera_position="iso"; pl.screenshot(str(PLOT_DIR/"vortex_structures"/f"q_isosurface_{k:02d}.png")); pl.close()
    print("Saved selected Q isosurfaces at one full-record threshold:",GLOBAL_Q_THRESHOLD)
except ImportError:
    print("PyVista unavailable; Q histories remain in the diagnostics CSV; no 3D isosurfaces generated.")

## 7. Subset animation and particle clouds

The animation uses all 150 reconstructed subset snapshots chronologically and fixed global color limits. Six particle cloud snapshots are used only for visualization; quantitative particle diagnostics above use the source snapshots through step 298. Marker colors represent `|Gamma|` and do not imply a turbulence measure.

In [ ]:
from matplotlib.animation import FuncAnimation, PillowWriter
# Stream one frame at a time for an all-snapshot animation.
fig,ax=plt.subplots(figsize=(10,4.5))
time0,x,y0,z,F0=read_grid_frame(field_files[0])
W0=F0["W"]; iy=int(np.argmin(np.abs(y0-span/2)))
# Stable global scales are the maxima across the six globally spaced selected times.
img=ax.pcolormesh(x,z,W0[:,iy,:,:][...,1],shading="auto",cmap="RdBu_r",vmin=-max_abs,vmax=max_abs)
title=ax.set_title(""); ax.set(xlabel="x (m)",ylabel="z (m)"); fig.colorbar(img,ax=ax,label="$\omega_y$ (1/s)")
def update(frame_i):
    t,xx,yy,zz,ff=read_grid_frame(field_files[frame_i]); iy0=int(np.argmin(np.abs(yy-span/2)))
    arr=ff["W"][:,iy0,:,:][...,1]
    img.set_array(arr.ravel()); title.set_text(f"Midspan signed vorticity, t={t:.5g} s ({frame_i+1}/{len(field_files)})")
    return img,title
ani=FuncAnimation(fig,update,frames=len(field_files),blit=False)
mp4=PLOT_DIR/"animations"/"midspan_omega_y_steps_0_298_stride_2.mp4"
try:
    ani.save(mp4,writer="ffmpeg",fps=15,dpi=120); print("Saved",mp4)
except Exception as exc:
    gif=PLOT_DIR/"animations"/"midspan_omega_y_steps_0_298_stride_2.gif"
    ani.save(gif,writer=PillowWriter(fps=10)); print("MP4 unavailable; saved",gif,"reason:",exc)
plt.close(fig)

# Particle clouds at first/25/50/75/final time. Free and static populations are shown separately.
for pop in ["pfield","staticpfield"]:
    pop_records=[r for r in records if r["population"]==pop and r["h5_exists"] and 0 <= int(r["step"]) <= POSTPROCESS_STEPS[-1]]
    if not pop_records: continue
    idxs=np.unique(np.rint(np.linspace(0,len(pop_records)-1,6)).astype(int))
    for j,ix in enumerate(idxs):
        r=pop_records[ix]
        with h5py.File(r["h5_path"],"r") as h:
            X=np.asarray(h[r["geometry_dataset"]]); G=np.asarray(h["Gamma"]) if "Gamma" in h else None
            sig=np.asarray(h["sigma"]).reshape(-1) if "sigma" in h else np.full(len(X),median_sigma)
        color=np.linalg.norm(G,axis=1) if G is not None and G.ndim==2 else np.zeros(len(X))
        marker_size=np.clip(8*sig/max(median_sigma,1e-30),1,24)
        fig=plt.figure(figsize=(8,6)); ax=fig.add_subplot(111,projection="3d")
        sc=ax.scatter(X[:,0],X[:,1],X[:,2],c=color,s=marker_size,cmap="viridis",vmin=0,vmax=max(particle_diag.Gamma_mag_max.max(),1e-30),rasterized=True)
        ax.set_xlim(BOUNDS[0]); ax.set_ylim(BOUNDS[1]); ax.set_zlim(BOUNDS[2])
        ax.set(xlabel="x (m)",ylabel="y (m)",zlabel="z (m)",title=f"{pop}, t={r['time_s']:.5g} s; color=|Gamma|")
        fig.colorbar(sc,ax=ax,label="|Gamma|"); fig.tight_layout(); fig.savefig(PLOT_DIR/"particle_visualization"/f"{pop}_{j:02d}.png",dpi=140); plt.close(fig)

## 8. Reynolds scales, coupled response and interpretation

The nominal Reynolds number is `V_inf * chord / nu` using the saved run configuration when available. The old supplied run does not include a configuration snapshot, so this notebook warns and reports the current-config fallback as unverified. It is not a turbulence-model or transition criterion by itself. Use structural displacement/rotation and aerodynamic load histories from the existing CSVs alongside flow diagnostics for coupled-response comparisons; association in time is evidence of correlation, not causality.

In [ ]:
# Configuration provenance and nominal Reynolds number (never silently assert historical settings).
run_cfg=RUN_DIR/"config"
config_provenance="saved run config" if run_cfg.is_dir() else "CURRENT repository config fallback; not verified as historical run settings"
if config_provenance.startswith("CURRENT"): warnings.warn(config_provenance)
fluid_cfg_used=yaml.safe_load((run_cfg/"fluid_params.yaml").read_text()) if (run_cfg/"fluid_params.yaml").exists() else yaml.safe_load(FLUID_CONFIG.read_text())
solid_cfg_used=yaml.safe_load((run_cfg/"solid_params.yaml").read_text()) if (run_cfg/"solid_params.yaml").exists() else yaml.safe_load(SOLID_CONFIG.read_text())
vinf=float(fluid_cfg_used.get("vinf",np.nan)); nu=float(fluid_cfg_used.get("nu",np.nan)); cref=max(float(solid_cfg_used.get("root_chord",np.nan)),float(solid_cfg_used.get("tip_chord",np.nan)))
print("Configuration source:",config_provenance,"Re_c:",vinf*cref/nu if nu>0 else np.nan)

# Align existing structural/coupling outputs by time for a compact comparison figure.
struct_csv=next(RUN_DIR.rglob("solid_state_history.csv"),None); couple_csv=next(RUN_DIR.rglob("coupling_history.csv"),None)
if struct_csv and couple_csv:
    sd=pd.read_csv(struct_csv); cd=pd.read_csv(couple_csv)
    fig,ax=plt.subplots(figsize=(10,4.5)); ax.plot(sd.time_s,sd.tip_w_displacement,label="tip w (m)")
    ax.set_xlabel("Time (s)"); ax.set_ylabel("tip displacement (m)"); ax.grid(alpha=.25)
    ax2=ax.twinx()
    if "lift" in cd: ax2.plot(cd.time,cd.lift,color="tab:red",alpha=.7,label="lift (N)")
    ax2.set_ylabel("lift (N)"); ax.set_title("Structural response and aerodynamic load (time-aligned)")
    fig.tight_layout(); fig.savefig(PLOT_DIR/"flow_diagnostics"/"structure_lift_time_history.png",dpi=160); plt.show()

print("Outputs saved under",ANALYSIS_DIR)
print("Inventory:",DATA_DIR/"particle_snapshot_inventory.csv")
print("Flow diagnostics:",DATA_DIR/"flow_field_diagnostics.csv")
print("Fixed Eulerian probes:",DATA_DIR/"eulerian_probe_history.csv")
print("Validation:",DATA_DIR/"reconstruction_validation.csv")